# Rice Image Dataset: CNN Classification (Educational Notebook)

Goal: train an explainable CNN to classify 4 rice varieties using TensorFlow, with scikit-learn for data splits and evaluation. Each step is documented so you can explain it during your exam.

## 1. Setup and imports

We set a fixed seed for reproducibility, then import the libraries used for data handling, model training, and evaluation.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from time import time

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_DIR = Path("Rice_Image_Dataset")
assert DATA_DIR.exists(), "Rice_Image_Dataset folder not found."


I0000 00:00:1780660904.461454 3788633 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1780660904.462120 3788633 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1780660904.699272 3788633 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780660906.290510 3788633 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONE

## 2. Dataset inspection

We list the class folders, count images per class, and confirm the dataset is unbalanced.

In [2]:
class_names = sorted([p.name for p in DATA_DIR.iterdir() if p.is_dir()])
class_names

['Arborio', 'Basmati', 'Ipsala', 'Jasmine']

In [3]:
file_paths = []
labels = []

for class_name in class_names:
    files = sorted((DATA_DIR / class_name).glob("*.jpg"))
    file_paths.extend([str(p) for p in files])
    labels.extend([class_name] * len(files))

df = pd.DataFrame({"path": file_paths, "label": labels})
class_to_index = {name: idx for idx, name in enumerate(class_names)}
df["label_id"] = df["label"].map(class_to_index)

df["label"].value_counts()

label
Arborio    15000
Basmati    15000
Ipsala     15000
Jasmine     6576
Name: count, dtype: int64

## 3. Train/validation/test split (70/15/15)

We use stratified splits to preserve class proportions in each subset.

In [4]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label_id"],
    random_state=SEED,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label_id"],
    random_state=SEED,
)

len(train_df), len(val_df), len(test_df)

(36103, 7736, 7737)

## 4. Input pipeline (tf.data)

We build a TensorFlow data pipeline to load images, resize them, normalize pixel values, and apply augmentation only to the training set. The images are resized from 250x250 to 224x224 to reduce compute while keeping enough detail.

In [6]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
    ]
)


def load_and_preprocess(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.image.convert_image_dtype(image, tf.float32)
    return image, label


def make_dataset(paths, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(buffer_size=len(paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(
            lambda x, y: (data_augmentation(x, training=True), y),
            num_parallel_calls=AUTOTUNE,
        )
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds


In [7]:
train_ds = make_dataset(train_df["path"].values, train_df["label_id"].values, training=True)
val_ds = make_dataset(val_df["path"].values, val_df["label_id"].values, training=False)
test_ds = make_dataset(test_df["path"].values, test_df["label_id"].values, training=False)

for images, labels in train_ds.take(1):
    print(images.shape, labels.shape)

(32, 224, 224, 3) (32,)


## 5. Class weights (handle imbalance)

Because one class has fewer samples, we compute class weights so the model pays more attention to minority classes during training.

In [8]:
num_classes = len(class_names)
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_classes),
    y=train_df["label_id"].values,
)
class_weight = dict(enumerate(class_weights))
class_weight

{0: np.float64(0.8595952380952381),
 1: np.float64(0.8595952380952381),
 2: np.float64(0.8595952380952381),
 3: np.float64(1.960840756028677)}

## 6. CNN architecture

We use a small, explainable CNN: a few convolution + pooling blocks for feature extraction, followed by dense layers for classification.

In [9]:
model = keras.Sequential(
    [
        layers.Input(shape=(*IMG_SIZE, 3)),
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Dropout(0.3),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(num_classes, activation="softmax"),
    ]
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()
cnn_params = model.count_params()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,476 (42.61 MB)

 Trainable params: 11,169,476 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

## 7. Training

We train with early stopping and checkpointing to reduce overfitting and keep the best model.

In [10]:
Path("models").mkdir(exist_ok=True)

callbacks = [
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("models/rice_cnn.keras", save_best_only=True),
]

start_time = time()
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    class_weight=class_weight,
    callbacks=callbacks,
)
cnn_train_time = time() - start_time

Epoch 1/20
  61/1129 ━━━━━━━━━━━━━━━━━━━━ 6:29 364ms/step - accuracy: 0.5648 - loss: 65.7913

KeyboardInterrupt: 

In [ ]:
def plot_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history["loss"], label="train")
    axes[0].plot(history.history["val_loss"], label="val")
    axes[0].set_title("Loss")
    axes[0].legend()

    axes[1].plot(history.history["accuracy"], label="train")
    axes[1].plot(history.history["val_accuracy"], label="val")
    axes[1].set_title("Accuracy")
    axes[1].legend()
    plt.show()


plot_history(history)

## 8. Evaluation

We evaluate the model on the test set and use a confusion matrix and classification report to understand per-class performance.

In [ ]:
cnn_test_loss, cnn_test_acc = model.evaluate(test_ds)
cnn_test_loss, cnn_test_acc

In [ ]:
y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_pred_probs = model.predict(test_ds)
y_pred = np.argmax(y_pred_probs, axis=1)

print(classification_report(y_true, y_pred, target_names=class_names))

ConfusionMatrixDisplay.from_predictions(
    y_true,
    y_pred,
    display_labels=class_names,
    cmap="Blues",
    xticks_rotation=45,
)

## 9. Vision Transformer (ViT) baseline

A ViT splits the image into fixed-size patches, projects each patch to an embedding vector, and uses transformer blocks (self-attention) instead of convolution for feature learning.

In [ ]:
PATCH_SIZE = 16
PROJECTION_DIM = 64
NUM_HEADS = 4
TRANSFORMER_LAYERS = 6
MLP_DIM = 128
VIT_DROPOUT = 0.1

NUM_PATCHES = (IMG_SIZE[0] // PATCH_SIZE) * (IMG_SIZE[1] // PATCH_SIZE)


def mlp(x, hidden_units, dropout_rate):
    for units in hidden_units:
        x = layers.Dense(units, activation=tf.nn.gelu)(x)
        x = layers.Dropout(dropout_rate)(x)
    return x


def build_vit():
    inputs = layers.Input(shape=(*IMG_SIZE, 3))
    patches = layers.Conv2D(
        PROJECTION_DIM,
        kernel_size=PATCH_SIZE,
        strides=PATCH_SIZE,
        padding="valid",
    )(inputs)
    x = layers.Reshape((NUM_PATCHES, PROJECTION_DIM))(patches)

    positions = tf.range(start=0, limit=NUM_PATCHES, delta=1)
    pos_embed = layers.Embedding(input_dim=NUM_PATCHES, output_dim=PROJECTION_DIM)(positions)
    x = x + pos_embed

    for _ in range(TRANSFORMER_LAYERS):
        x1 = layers.LayerNormalization(epsilon=1e-6)(x)
        attention_output = layers.MultiHeadAttention(
            num_heads=NUM_HEADS,
            key_dim=PROJECTION_DIM,
            dropout=VIT_DROPOUT,
        )(x1, x1)
        x2 = layers.Add()([attention_output, x])
        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        x3 = mlp(x3, hidden_units=[MLP_DIM, PROJECTION_DIM], dropout_rate=VIT_DROPOUT)
        x = layers.Add()([x3, x2])

    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return keras.Model(inputs, outputs, name="vit_baseline")


In [ ]:
vit_model = build_vit()
vit_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
vit_params = vit_model.count_params()
vit_model.summary()

## 10. ViT training

We keep the same data split and class weights for a fair comparison.

In [ ]:
vit_callbacks = [
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("models/rice_vit.keras", save_best_only=True),
]

start_time = time()
vit_history = vit_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weight,
    callbacks=vit_callbacks,
)
vit_train_time = time() - start_time

In [ ]:
plot_history(vit_history)

## 11. ViT evaluation

We evaluate the ViT on the same test set and compare per-class performance.

In [ ]:
vit_test_loss, vit_test_acc = vit_model.evaluate(test_ds)
vit_test_loss, vit_test_acc

In [ ]:
vit_y_true = np.concatenate([y.numpy() for _, y in test_ds])
vit_y_pred_probs = vit_model.predict(test_ds)
vit_y_pred = np.argmax(vit_y_pred_probs, axis=1)

print(classification_report(vit_y_true, vit_y_pred, target_names=class_names))

ConfusionMatrixDisplay.from_predictions(
    vit_y_true,
    vit_y_pred,
    display_labels=class_names,
    cmap="Blues",
    xticks_rotation=45,
)

## 12. CNN vs ViT comparison

We summarize accuracy, loss, parameter count, and training time for a direct comparison.

In [ ]:
comparison = pd.DataFrame(
    [
        {
            "model": "CNN",
            "params": cnn_params,
            "test_acc": cnn_test_acc,
            "test_loss": cnn_test_loss,
            "train_time_sec": cnn_train_time,
        },
        {
            "model": "ViT",
            "params": vit_params,
            "test_acc": vit_test_acc,
            "test_loss": vit_test_loss,
            "train_time_sec": vit_train_time,
        },
    ]
)
comparison

## 13. Wrap-up and next steps

- Explain which classes are easiest/hardest and why.
- Discuss overfitting signals from the training curves for CNN vs ViT.
- Try improvements: more augmentation, deeper CNN, larger ViT, or fine-tuning a pre-trained model.

This notebook is intentionally simple so you can fully explain each step during your exam.